In [0]:
# ============================================================
# NOTEBOOK: nb_06_Validation
# PURPOSE:  Consolidates data-quality and reconciliation checks
#           across every layer into a single results table.
#           A reviewer opens one table to see whether the
#           pipeline can be trusted.
#
# OUTPUTS:
#   gold.validation_results  - one row per check
#
# CATALOG:  ktu_assessment_dev
# COMPUTE:  Serverless
# ============================================================

import uuid
from datetime import datetime
from pyspark.sql import functions as F

CATALOG       = "ktu_assessment_dev"
AUDIT_SCHEMA  = "audit"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA   = "gold"

DEBUG = 1

run_id     = str(uuid.uuid4())
notebook   = "nb_06_Validation"
start_time = datetime.now()

if DEBUG:
    print("=" * 50)
    print("NB_06_VALIDATION STARTED")
    print("=" * 50)
    print(f"Run ID     : {run_id}")
    print(f"Start Time : {start_time}")

NB_06_VALIDATION STARTED
Run ID     : aa83c1bc-03d7-4c6a-8b50-f926febcdcf4
Start Time : 2026-09-12 14:55:20.088856


In [0]:
# ============================================================
# VALIDATION HELPER
# ============================================================
#
# WHAT THIS CELL DOES:
# Defines a helper that appends a check row to a list and a
# runner that writes the accumulated rows to a Delta table.
#
# COLUMNS:
#   run_id        - pipeline run that produced the check
#   category      - completeness, uniqueness, reconciliation, integrity, accounting
#   check_name    - short identifier
#   description   - what the check tests
#   expected      - expected value, human-readable
#   actual        - actual value, human-readable
#   status        - PASS or FAIL
#   message       - additional detail when status is FAIL
# ============================================================

validation_rows = []


def add_check(category, check_name, description, expected, actual, message=None):
    """Append a validation result row to the accumulator."""
    status = "PASS" if str(expected) == str(actual) else "FAIL"
    validation_rows.append((
        run_id, category, check_name, description,
        str(expected), str(actual), status, message
    ))
    if DEBUG:
        marker = "PASS" if status == "PASS" else "FAIL"
        print(f"  [{marker}] {check_name:<45} expected={expected}  actual={actual}")


if DEBUG:
    print("Validation accumulator ready.")

Validation accumulator ready.


In [0]:
# ============================================================
# VALIDATION CHECKS
# ============================================================
#
# All expected values for reconciliation checks are computed
# from the dedup fact itself, not hard-coded. A hard-coded
# expected value becomes stale whenever an upstream change
# alters the row population. Computing it guarantees the
# check is testing the pipeline's consistency, not its
# coincidence with a number captured at an earlier point
# in time.
# ============================================================

if DEBUG:
    print("=" * 70)
    print("RUNNING VALIDATION CHECKS")
    print("=" * 70)

# ------------------------------------------------------------
# CHECK 1: BRONZE ROW COUNTS
# ------------------------------------------------------------
bronze_counts = {}
for t in ["capturing_tool", "chw_attendance", "online_export",
          "lookup_courses", "lookup_facility"]:
    cnt = spark.sql(f"SELECT COUNT(*) AS n FROM {CATALOG}.{BRONZE_SCHEMA}.{t}").collect()[0]["n"]
    bronze_counts[t] = cnt

add_check("completeness", "bronze_capturing_tool_rows",
          "Bronze capturing_tool row count", 11607, bronze_counts["capturing_tool"])
add_check("completeness", "bronze_chw_attendance_rows",
          "Bronze chw_attendance row count", 359, bronze_counts["chw_attendance"])
add_check("completeness", "bronze_online_export_rows",
          "Bronze online_export row count", 25824, bronze_counts["online_export"])
add_check("completeness", "bronze_lookup_courses_rows",
          "Bronze lookup_courses row count", 340, bronze_counts["lookup_courses"])
add_check("completeness", "bronze_lookup_facility_rows",
          "Bronze lookup_facility row count", 809, bronze_counts["lookup_facility"])

# ------------------------------------------------------------
# CHECK 2: SILVER FACT ROW COUNT
# ------------------------------------------------------------
silver_count = spark.sql(
    f"SELECT COUNT(*) AS n FROM {CATALOG}.{SILVER_SCHEMA}.participant_event"
).collect()[0]["n"]

expected_silver = (
    bronze_counts["capturing_tool"] +
    bronze_counts["chw_attendance"] +
    bronze_counts["online_export"]
)

add_check("completeness", "silver_fact_row_count",
          "Silver participant_event = sum of three fact sources",
          expected_silver, silver_count)

# ------------------------------------------------------------
# DEDUP FACT HEADLINE VALUES (computed, not hard-coded)
# ------------------------------------------------------------
dedup_stats = spark.sql(f"""
    SELECT
        COUNT(*) AS enrolments,
        SUM(CASE WHEN is_completed THEN 1 ELSE 0 END) AS completions,
        COUNT(DISTINCT participant_key) AS unique_participants
    FROM {CATALOG}.{SILVER_SCHEMA}.participant_event_dedup
""").collect()[0]

dedup_count       = int(dedup_stats["enrolments"])
expected_complete = int(dedup_stats["completions"])

# ------------------------------------------------------------
# CHECK 3: DEDUP AND EXCLUDED ACCOUNTING
# ------------------------------------------------------------
excluded_count = spark.sql(
    f"SELECT COUNT(*) AS n FROM {CATALOG}.{SILVER_SCHEMA}.excluded_records"
).collect()[0]["n"]

add_check("accounting", "silver_to_dedup_plus_excluded",
          "Silver fact = dedup fact + excluded records",
          silver_count, dedup_count + excluded_count)

# ------------------------------------------------------------
# CHECK 4: DEDUP FACT ROWS ARE Q4 2025
# ------------------------------------------------------------
out_of_scope = spark.sql(f"""
    SELECT COUNT(*) AS n
    FROM {CATALOG}.{SILVER_SCHEMA}.participant_event_dedup
    WHERE event_date < '2025-10-01' OR event_date > '2025-12-31'
""").collect()[0]["n"]

add_check("integrity", "dedup_all_in_q4_2025",
          "Every dedup fact row falls in Q4 2025", 0, out_of_scope)

# ------------------------------------------------------------
# CHECK 5: NO EVENT DOUBLE-COUNTED BETWEEN DEDUP AND EXCLUDED
# ------------------------------------------------------------
overlap = spark.sql(f"""
    SELECT COUNT(*) AS n
    FROM (
        SELECT event_key FROM {CATALOG}.{SILVER_SCHEMA}.participant_event_dedup
        INTERSECT
        SELECT event_key FROM {CATALOG}.{SILVER_SCHEMA}.excluded_records
    )
""").collect()[0]["n"]

add_check("uniqueness", "no_event_key_overlap",
          "No event_key appears in both dedup fact and excluded records",
          0, overlap)

# ------------------------------------------------------------
# CHECK 6: REPORT 1 SUBTOTALS EQUAL HEADLINE
# ------------------------------------------------------------
district_sum = spark.sql(f"""
    SELECT SUM(enrolments) AS e, SUM(completions) AS c
    FROM {CATALOG}.{GOLD_SCHEMA}.report_completions_by_district
    WHERE district <> 'TOTAL'
""").collect()[0]

add_check("reconciliation", "district_enrolment_subtotals",
          "District report enrolments subtotal = dedup fact count",
          dedup_count, int(district_sum["e"]))

add_check("reconciliation", "district_completion_subtotals",
          "District report completions subtotal = dedup fact completions",
          expected_complete, int(district_sum["c"]))

# ------------------------------------------------------------
# CHECK 7: REPORT 2 SUBTOTALS EQUAL HEADLINE
# ------------------------------------------------------------
source_sum = spark.sql(f"""
    SELECT SUM(enrolments) AS e, SUM(completions) AS c
    FROM {CATALOG}.{GOLD_SCHEMA}.report_enrolments_vs_completions
    WHERE source_system <> 'TOTAL'
""").collect()[0]

add_check("reconciliation", "source_enrolment_subtotals",
          "Source report enrolments subtotal = dedup fact count",
          dedup_count, int(source_sum["e"]))

add_check("reconciliation", "source_completion_subtotals",
          "Source report completions subtotal = dedup fact completions",
          expected_complete, int(source_sum["c"]))

# ------------------------------------------------------------
# CHECK 8: ENROLMENTS = COMPLETIONS + NOT_COMPLETED
# ------------------------------------------------------------
add_check("reconciliation", "enrolments_split_correctly",
          "Enrolments = completions + not_completed",
          dedup_count, expected_complete + (dedup_count - expected_complete))

# ------------------------------------------------------------
# CHECK 9: RECONCILIATION RESULTS TABLE FROM GOLD
# ------------------------------------------------------------
gold_fails = spark.sql(f"""
    SELECT COUNT(*) AS n FROM {CATALOG}.{GOLD_SCHEMA}.reconciliation_results
    WHERE status = 'FAIL'
""").collect()[0]["n"]

add_check("reconciliation", "gold_reconciliation_all_pass",
          "Gold reconciliation_results contains zero FAIL rows",
          0, gold_fails)

# ------------------------------------------------------------
# CHECK 10: UNMAPPED VALUES ARE SURFACED
# ------------------------------------------------------------
unmapped_fac = spark.sql(
    f"SELECT COUNT(*) AS n FROM {CATALOG}.{SILVER_SCHEMA}.unmapped_facilities"
).collect()[0]["n"]

add_check("referential", "unmapped_facilities_surfaced",
          "Unmapped facility values are recorded (not dropped)",
          ">0", ">0" if unmapped_fac > 0 else "0",
          f"{unmapped_fac} distinct unmapped facility values recorded")

# ------------------------------------------------------------
# CHECK 11: AUDIT LOG ENTRIES FOR EVERY NOTEBOOK
# ------------------------------------------------------------
notebooks_in_audit = spark.sql(f"""
    SELECT DISTINCT notebook_name
    FROM {CATALOG}.{AUDIT_SCHEMA}.pipeline_run_log
    WHERE status = 'SUCCESS'
""").collect()

notebook_names = sorted([r["notebook_name"] for r in notebooks_in_audit])
expected_notebooks = ["nb_00_Setup", "nb_01_Ingestion", "nb_02_Bronze",
                      "nb_03_Silver", "nb_04_Deduplication", "nb_05_Gold"]

missing = [n for n in expected_notebooks if n not in notebook_names]

add_check("integrity", "audit_covers_all_notebooks",
          "Every pipeline notebook has a SUCCESS audit entry",
          0, len(missing),
          f"Missing: {missing}" if missing else None)

RUNNING VALIDATION CHECKS
  [PASS] bronze_capturing_tool_rows                    expected=11607  actual=11607
  [PASS] bronze_chw_attendance_rows                    expected=359  actual=359
  [PASS] bronze_online_export_rows                     expected=25824  actual=25824
  [PASS] bronze_lookup_courses_rows                    expected=340  actual=340
  [PASS] bronze_lookup_facility_rows                   expected=809  actual=809
  [PASS] silver_fact_row_count                         expected=37790  actual=37790
  [PASS] silver_to_dedup_plus_excluded                 expected=37790  actual=37790
  [PASS] dedup_all_in_q4_2025                          expected=0  actual=0
  [PASS] no_event_key_overlap                          expected=0  actual=0
  [PASS] district_enrolment_subtotals                  expected=4832  actual=4832
  [PASS] district_completion_subtotals                 expected=3850  actual=3850
  [PASS] source_enrolment_subtotals                    expected=4832  actual=4832


In [0]:
# ============================================================
# WRITE VALIDATION RESULTS
# ============================================================

schema = (
    "run_id STRING, category STRING, check_name STRING, "
    "description STRING, expected STRING, actual STRING, "
    "status STRING, message STRING"
)

validation_df = spark.createDataFrame(validation_rows, schema=schema)

spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{GOLD_SCHEMA}.validation_results")

validation_df.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.validation_results")

if DEBUG:
    print()
    print("=" * 70)
    print("VALIDATION RESULTS TABLE")
    print("=" * 70)
    validation_df.show(50, truncate=False)


VALIDATION RESULTS TABLE
+------------------------------------+--------------+-----------------------------+-------------------------------------------------------------+--------+------+------+----------------------------------------------+
|run_id                              |category      |check_name                   |description                                                  |expected|actual|status|message                                       |
+------------------------------------+--------------+-----------------------------+-------------------------------------------------------------+--------+------+------+----------------------------------------------+
|aa83c1bc-03d7-4c6a-8b50-f926febcdcf4|completeness  |bronze_capturing_tool_rows   |Bronze capturing_tool row count                              |11607   |11607 |PASS  |NULL                                          |
|aa83c1bc-03d7-4c6a-8b50-f926febcdcf4|completeness  |bronze_chw_attendance_rows   |Bronze chw_attendance row c

In [0]:
# ============================================================
# VALIDATION SUMMARY
# ============================================================

total  = len(validation_rows)
passed = sum(1 for r in validation_rows if r[6] == "PASS")
failed = total - passed

if DEBUG:
    print("=" * 70)
    print("VALIDATION SUMMARY")
    print("=" * 70)
    print(f"Total checks : {total}")
    print(f"Passed       : {passed}")
    print(f"Failed       : {failed}")
    print()

    if failed > 0:
        print("FAILED CHECKS:")
        for r in validation_rows:
            if r[6] == "FAIL":
                print(f"  {r[2]}  expected={r[4]}  actual={r[5]}")
        print()
        print("Review the validation_results table for details.")

    print()
    print("DISTRIBUTION BY CATEGORY:")
    spark.sql(f"""
        SELECT category, status, COUNT(*) AS checks
        FROM {CATALOG}.{GOLD_SCHEMA}.validation_results
        GROUP BY category, status
        ORDER BY category, status
    """).show(truncate=False)

    print("=" * 70)

VALIDATION SUMMARY
Total checks : 17
Passed       : 17
Failed       : 0


DISTRIBUTION BY CATEGORY:
+--------------+------+------+
|category      |status|checks|
+--------------+------+------+
|accounting    |PASS  |1     |
|completeness  |PASS  |6     |
|integrity     |PASS  |2     |
|reconciliation|PASS  |6     |
|referential   |PASS  |1     |
|uniqueness    |PASS  |1     |
+--------------+------+------+



In [0]:
# ============================================================
# FINALISE AUDIT
# ============================================================

end_time = datetime.now()
duration = int((end_time - start_time).total_seconds())

status = "SUCCESS" if failed == 0 else "FAILED"

spark.sql(f"DELETE FROM {CATALOG}.{AUDIT_SCHEMA}.data_quality_results WHERE run_id = '{run_id}'")

spark.sql(f"""
    INSERT INTO {CATALOG}.{AUDIT_SCHEMA}.data_quality_results
    VALUES (
        '{run_id}',
        'consolidated_validation',
        'validation',
        'gold.validation_results',
        'All checks PASS',
        '{passed} PASS, {failed} FAIL',
        '{ "PASS" if failed == 0 else "FAIL" }',
        NULL,
        current_timestamp()
    )
""")

spark.sql(f"""
    UPDATE {CATALOG}.{AUDIT_SCHEMA}.pipeline_run_log
    SET
        end_time         = '{end_time.strftime("%Y-%m-%d %H:%M:%S")}',
        status           = '{status}',
        rows_in          = {total},
        rows_out         = {passed},
        rows_rejected    = {failed},
        message          = 'Validation complete. {passed} PASS, {failed} FAIL.',
        duration_seconds = {duration}
    WHERE run_id = '{run_id}'
""")

if DEBUG:
    print()
    print("=" * 50)
    print("VALIDATION NOTEBOOK SUMMARY")
    print("=" * 50)
    print(f"Run ID      : {run_id}")
    print(f"Total       : {total}")
    print(f"Passed      : {passed}")
    print(f"Failed      : {failed}")
    print(f"Duration    : {duration}s")
    print(f"Status      : {status}")
    print("=" * 50)

print("nb_06_Validation completed successfully.")


VALIDATION NOTEBOOK SUMMARY
Run ID      : aa83c1bc-03d7-4c6a-8b50-f926febcdcf4
Total       : 17
Passed      : 17
Failed      : 0
Duration    : 10s
Status      : SUCCESS
nb_06_Validation completed successfully.


In [0]:
# ============================================================
# FAILED CHECK DETAIL
# ============================================================

spark.sql(f"""
    SELECT check_name, description, expected, actual, status, message
    FROM {CATALOG}.{GOLD_SCHEMA}.validation_results
    WHERE status = 'FAIL'
""").show(truncate=False)

+----------+-----------+--------+------+------+-------+
|check_name|description|expected|actual|status|message|
+----------+-----------+--------+------+------+-------+
+----------+-----------+--------+------+------+-------+

